In [1]:
# Run this, then go to "Kernel" > "Restart" in your notebook menu
!python -m pip install scikit-learn==1.5.2


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import seaborn as sns
import matplotlib.pyplot as plt
import os  # <--- ADDED THIS MISSING IMPORT

# 1. LOAD DATA
if not os.path.exists('data/processed/training_data_final.csv'):
    print("❌ Run Notebook 02 first!")
    exit()

df = pd.read_csv('data/processed/training_data_final.csv')
features = ['latitude', 'longitude', 'month_sin', 'month_cos', 'temp', 'humidity', 'wind', 'vpd']
X = df[features]; y = df['fire_detected']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. DEFINE MODELS
print("🧠 Initializing Models...")
cat = CatBoostClassifier(iterations=1000, depth=8, learning_rate=0.05, verbose=0)
xg = xgb.XGBClassifier(n_estimators=1000, max_depth=8, learning_rate=0.05)

ensemble = VotingClassifier(estimators=[('cat', cat), ('xg', xg)], voting='soft')

# 3. TRAIN
print("🤖 Fitting Ensemble (This may take 1-2 minutes)...")
ensemble.fit(X_train, y_train)

# 4. EVALUATE
print("\n🏆 Final Metrics:")
print(classification_report(y_test, ensemble.predict(X_test)))

# 5. SAVE
os.makedirs('models', exist_ok=True)
joblib.dump(ensemble, 'models/wildfire_ensemble.pkl')
print("✅ Model Saved: models/wildfire_ensemble.pkl")

🧠 Initializing Models...
🤖 Fitting Ensemble (This may take 1-2 minutes)...

🏆 Final Metrics:
              precision    recall  f1-score   support

           0       0.83      0.74      0.78     20030
           1       0.76      0.85      0.80     19964

    accuracy                           0.79     39994
   macro avg       0.80      0.79      0.79     39994
weighted avg       0.80      0.79      0.79     39994

✅ Model Saved: models/wildfire_ensemble.pkl
